# A0 — `C.—.— → Ut`: untuned CompoundT5 on the canonical USPTO-50K test

The zero point of the clean benchmark: what the base checkpoint scores with no fine-tuning at
all. On the ORD test the same checkpoint produced 0.0% exact with 14.7% parseable SMILES, so a
near-zero result is expected here too -- the value of this run is that every later number in the
A-series is stated as a gain over a measured baseline rather than over an assumption.


**Benchmark.** `bisectgroup/USPTO_50K`, the canonical 40008/5001/5007 split used across the
retrosynthesis literature. Training rows come from its `train` partition, the test set from its
`test` partition (5003 unique products; 82 rows whose product also occurs in `test` were dropped
from the training side, so the overlap is exactly zero). The base `sagawa/CompoundT5` was
pretrained by span-MLM over 24M ZINC20 *molecules* and has never seen a reaction, so nothing in
this benchmark can leak through pretraining -- unlike the ORD line, where
`ReactionT5v2-retrosynthesis` was pretrained on ~1.5M ORD reactions and the ORD test is drawn
from the same database.

**Reference points (literature, same split):** R-SMILES 56.3% top-1 / 86.2% top-5 exact;
RetroKNN 57.2% top-1.


**Cost:** no training, ~35 min for the 1000-record beam-10 evaluation.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
# Comparison set: 1000 records, a strict prefix of the full 5003-record canonical test
# split (same seed), so numbers here and on the full set are drawn from one distribution.
model_dir = "sagawa/CompoundT5"
!python scripts/models/run_reactiont5_topk.py \
    --input "data/v2_uspto_test_holdout_1000.json" --t5-model "{model_dir}" \
    --num-beams 10 --device cuda \
    --output "/kaggle/working/A0_base_uspto1000_topk.json"

In [ ]:
import json
data = json.load(open("/kaggle/working/A0_base_uspto1000_topk.json"))
print(json.dumps(data["summary"], indent=2))